# Personal Knowledge Worker with RAG

A conversational AI that answers questions about your personal documents using Retrieval-Augmented Generation (RAG) with Chroma as the vector datastore.

**Week 5 Community Contribution by Mirack**

Skills demonstrated: RAG pipeline, Chroma vector store, document chunking, embeddings, conversational Q&A.

In [ ]:
import os
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions
import glob

In [ ]:

client = OpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.environ.get("OPENAI_API_KEY")
)

MODEL = "gpt-4.1-nano"

## 1. Create Sample Knowledge Base
In a real scenario you'd load your own files. Here we create sample documents to demonstrate the pipeline.

In [ ]:
# Sample documents simulating a personal knowledge base
documents = [
    {
        "title": "Project Alpha - Status Update",
        "content": """Project Alpha is a machine learning pipeline for customer churn prediction. 
        The team consists of 4 engineers. We are using XGBoost and LightGBM as our primary models.
        Current accuracy is 87% on the test set. The deadline is March 30, 2026.
        Key blocker: we need more training data from the marketing team.
        Next milestone: deploy to staging by March 15."""
    },
    {
        "title": "Meeting Notes - Sprint Review",
        "content": """Sprint review held on Feb 20, 2026. Attendees: Mirack, Sarah, Tom, Lisa.
        Completed: API integration with payment provider, user dashboard redesign.
        In progress: notification system, performance optimization.
        Action items: Mirack to set up monitoring dashboards by Feb 28.
        Sarah to finalize the API documentation. Tom to fix the caching bug."""
    },
    {
        "title": "Learning Notes - LLM Engineering",
        "content": """Week 1-3 covered API integration with OpenAI, Anthropic, and Gemini.
        Key concepts: streaming responses, JSON structured output, tokenization.
        HuggingFace transformers library is essential for working with open source models.
        The GPT-2 tokenizer has a vocabulary of 50,257 tokens.
        RAG combines retrieval from a knowledge base with LLM generation for grounded answers."""
    },
    {
        "title": "Personal Goals 2026",
        "content": """Career: Complete LLM Engineering course and build 3 portfolio projects.
        Technical: Learn Docker, Kubernetes, and deploy at least 2 ML models to production.
        Health: Run a half marathon by September. Gym 4 times per week.
        Finance: Save 20% of monthly income. Start investing in index funds.
        Reading: Finish 12 books this year, at least 4 technical."""
    },
    {
        "title": "Tech Stack Reference",
        "content": """Backend: Python, FastAPI, PostgreSQL, Redis for caching.
        Frontend: React, TypeScript, TailwindCSS.
        ML/AI: PyTorch, HuggingFace, LangChain, ChromaDB.
        DevOps: Docker, GitHub Actions, AWS (EC2, S3, Lambda).
        Monitoring: Prometheus, Grafana. Logging: ELK stack."""
    }
]

print(f"Loaded {len(documents)} documents into knowledge base")

## 2. Chunk and Vectorize with Chroma

In [ ]:
def chunk_document(doc, chunk_size=200):
    """Split a document into smaller chunks for better retrieval."""
    words = doc["content"].split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk_text = " ".join(words[i:i + chunk_size])
        chunks.append({
            "text": chunk_text,
            "source": doc["title"]
        })
    return chunks

# Chunk all documents
all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_document(doc))

print(f"Created {len(all_chunks)} chunks from {len(documents)} documents")

In [ ]:
# Initialize Chroma with default embedding function
chroma_client = chromadb.Client()

# Create or reset collection
collection = chroma_client.get_or_create_collection(
    name="knowledge_base",
    metadata={"description": "Personal knowledge worker documents"}
)

# Add chunks to Chroma
collection.add(
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"source": c["source"]} for c in all_chunks],
    ids=[f"chunk_{i}" for i in range(len(all_chunks))]
)

print(f"Indexed {collection.count()} chunks in Chroma")

## 3. RAG Query Pipeline
Retrieve relevant chunks, then pass them to the LLM for a grounded answer.

In [ ]:
def query_knowledge_base(question, n_results=3):
    """Search the vector store for relevant chunks."""
    results = collection.query(
        query_texts=[question],
        n_results=n_results
    )
    return results

def ask_knowledge_worker(question, n_results=3):
    """Full RAG pipeline: retrieve context then generate answer with streaming."""
    # Retrieve
    results = query_knowledge_base(question, n_results)
    context_chunks = results["documents"][0]
    sources = results["metadatas"][0]
    
    context = "\n\n".join([f"[{s['source']}]: {c}" for c, s in zip(context_chunks, sources)])
    
    # Generate
    prompt = f"""Answer the question based on the context below. 
If the context doesn't contain the answer, say so honestly.

CONTEXT:
{context}

QUESTION: {question}"""

    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful knowledge assistant. Answer based only on the provided context. Cite which document your answer comes from."},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )
    
    print(f"Sources: {', '.join([s['source'] for s in sources])}\n")
    
    result = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="")
            result += content
    return result

In [ ]:
print("Q: What is the deadline for Project Alpha?\n")
answer1 = ask_knowledge_worker("What is the deadline for Project Alpha?")

In [ ]:
print("Q: What are Mirack's action items from the sprint review?\n")
answer2 = ask_knowledge_worker("What are Mirack's action items from the sprint review?")

In [ ]:
print("Q: What tech stack am I using for backend?\n")
answer3 = ask_knowledge_worker("What tech stack am I using for backend?")

In [ ]:
# Test with a question that isn't in the knowledge base
print("Q: What is the weather today?\n")
answer4 = ask_knowledge_worker("What is the weather today?")

## 4. Conversational RAG
Add chat history so follow-up questions work naturally.

In [ ]:
def chat_with_knowledge(question, chat_history=None, n_results=3):
    """RAG with conversation memory for follow-up questions."""
    if chat_history is None:
        chat_history = []
    
    results = query_knowledge_base(question, n_results)
    context_chunks = results["documents"][0]
    sources = results["metadatas"][0]
    context = "\n\n".join([f"[{s['source']}]: {c}" for c, s in zip(context_chunks, sources)])
    
    messages = [
        {"role": "system", "content": "You are a knowledge assistant. Answer from the context. Cite sources. Handle follow-ups using chat history."}
    ]
    # Add chat history
    for h, a in chat_history:
        messages.append({"role": "user", "content": h})
        messages.append({"role": "assistant", "content": a})
    
    messages.append({"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"})
    
    stream = client.chat.completions.create(model=MODEL, messages=messages, stream=True)
    
    result = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="")
            result += content
    
    chat_history.append((question, result))
    return result, chat_history

In [ ]:
# Multi-turn conversation
history = []

print("Q1: What ML models are we using for Project Alpha?\n")
a1, history = chat_with_knowledge("What ML models are we using for Project Alpha?", history)

print("\n\nQ2: What's the current accuracy?\n")
a2, history = chat_with_knowledge("What's the current accuracy?", history)

print("\n\nQ3: What's blocking progress?\n")
a3, history = chat_with_knowledge("What's blocking progress?", history)